# Notebook 2: Training and evaluation (pipelines K1 to K9, R1 to R9, F)

Trains the **same model with the same settings** on each preprocessing pipeline and records the scores. Only the preprocessing changes between pipelines, so score differences can be attributed to preprocessing.

## Three scenarios, one notebook

Set `SCENARIO` in section 2 and commit. Each scenario writes to its own folder (`results/all`, `results/single`, `results/multi`), so the three can run one after another on one account, or in parallel on co-authors' accounts.

| `SCENARIO` | Images used | Extra outputs |
|---|---|---|
| `"all"` | single and multi together | **cross-device run**: the best model on F trained on iPhone images only, tested on the iPhone test images and on every image from the other phones |
| `"single"` | single-weapon images only | **cross-test**: every model also predicts the **multi** test images |
| `"multi"` | multi-weapon images only | **cross-test**: every model also predicts the **single** test images |

All scenarios use the same locked test set and folds from Stage 0, so their results are directly comparable.

## Inputs (attach with Add Input)

| Input | Where it comes from | Used for |
|---|---|---|
| **Stage 0 output** | your saved Stage 0 notebook (use the **same** one for every commit, scenario and Notebook 3) | `manifest.csv` (labels, subset, device, test set, folds) and the images **with background** (K pipelines) |
| **Notebook 1 output** | the saved background-removal notebook | the images **without background** (R pipelines) |
| *(from commit 2 on)* **this notebook's own output** | the previous saved version of this notebook, same scenario | earlier results are carried forward, so the final version holds everything |

## The pipelines

Reference = background **kept**, **letterbox 224**, **light** augmentation, **RGB**. Each pipeline changes one factor.

| Kept | Removed | Change from reference |
|---|---|---|
| K1 | R1 | none (reference) |
| K2 | R2 | size: stretch 224 |
| K3 | R3 | size: letterbox 384 |
| K4 | R4 | augmentation: none |
| K5 | R5 | augmentation: strong (RandAugment + random erasing + CutMix) |
| K6 | R6 | color: grayscale |
| K7 | R7 | color: HSV |
| K8 | R8 | color: LAB |
| K9 | R9 | color: CLAHE-enhanced RGB |
| F | | best level of each factor + class-balanced sampling + test-time augmentation, chosen automatically |

## How to run: set `SCENARIO`, then commit until it says "ALL DONE"

The notebook works through a fixed plan and skips anything already finished:

1. ResNet-50 on K1 to K9 and R1 to R9 (the preprocessing study)
2. ResNet-50 on **F**. F is chosen **automatically** from step 1: the best background, size, augmentation and color by mean CV macro-F1 (saved to `f_settings.json`)
3. ConvNeXt-Tiny, EfficientNet-B0 and ViT-S/16, each on F, K1 and R1
4. *(scenario `all` only)* the best model on F, trained on iPhone images only: background kept (`F_iphone`), background removed (`F_iphone_bgremoved`), and with camera-simulation augmentation (`F_iphone_camaug`)
5. Foundation-model baseline: frozen DINOv2 ViT-S/14 features + logistic regression, on the scenario's folds (`LP`) and, in scenario `all`, trained on iPhone images only (`LP_iphone`). No fine-tuning, a few minutes.

`TIME_BUDGET_HOURS` (default 10) stops starting new pipelines before Kaggle's 12 h limit. A commit that hits the limit saves nothing, so this guard matters.

| Commit | Inputs | What happens |
|---|---|---|
| 1 | Stage 0 + Notebook 1 | runs the plan until about 10 h |
| 2 | Stage 0 + Notebook 1 + **this notebook (version 1)** | carries results forward and continues |
| 3 (if needed) | same as 2, with this notebook's latest version | finishes whatever is left |

Rough GPU time (P100/T4): `all` about 14 to 16 h, `single` about 9 h, `multi` about 6 h.

**Evaluation protocol:** for each pipeline, 5 models are trained (one per fold). Each fold model is scored on its own validation fold at the final epoch (no epoch picking, so the score is unbiased). The held-out test set is predicted by averaging the 5 fold models.

## 1. Setup

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])

import os, json, time, math, random, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F_
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T
import timm
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__, "| timm", timm.__version__)
if DEVICE == "cpu":
    warnings.warn("No GPU found. Turn on Accelerator: GPU (T4 or P100) in the notebook settings.")

device: cuda | torch 2.11.0+cu128 | timm 1.0.29


## 2. Configuration (edit this cell for each run)

In [2]:
# ---- which analysis this commit runs: "all", "single" or "multi" ----
SCENARIO = "multi"

# ---- the plan: nothing to edit. Each commit continues where the previous one stopped ----
STAGE1_MODEL  = "resnet50"
STAGE1_PIPES  = [f"K{i}" for i in range(1, 10)] + [f"R{i}" for i in range(1, 10)] + ["F"]
STAGE2_MODELS = ["convnext_tiny", "efficientnet_b0", "vit_small_patch16_224"]
STAGE2_PIPES  = ["F", "K1", "R1"]
PLAN = [(STAGE1_MODEL, p) for p in STAGE1_PIPES] + [(m, p) for m in STAGE2_MODELS for p in STAGE2_PIPES]
TIME_BUDGET_HOURS = 10.0  # no new pipeline is started if it would likely end past this (Kaggle hard limit: 12 h)
MODELS = [STAGE1_MODEL] + STAGE2_MODELS
CROSS_DEVICE_TRAIN = "iPhone 14"   # scenario "all": the F_iphone run trains on this phone only
SAVE_ALL_FOLD_WEIGHTS = ["F"]      # pipelines whose 5 fold models are all saved (robustness tests in Notebook 3)

# ---- model and training settings: keep IDENTICAL for all pipelines ----
MODEL_NAME   = STAGE1_MODEL    # set per step by the run loop
PRETRAINED   = True
EPOCHS       = 25
BATCH_SIZE   = 32
LR           = 2e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 1
LABEL_SMOOTHING = 0.1
FOLDS        = [0, 1, 2, 3, 4]
SEED         = 42
NUM_WORKERS  = 4

# ---- final pipeline F: None = choose automatically from the ResNet-50 K/R results ----
# (to force a choice instead: F_SETTINGS = dict(bg="kept", size="letterbox", res=384, aug="light", color="rgb"))
F_SETTINGS = None

# ---- paths ----
ON_KAGGLE = Path("/kaggle/input").is_dir()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path(r"H:\BD Weapon Inputs")
OUT_ROOT   = Path("/kaggle/working") if ON_KAGGLE else Path(r"H:\BD Weapon Training")
assert SCENARIO in ("all", "single", "multi"), 'SCENARIO must be "all", "single" or "multi"'
RESULTS    = OUT_ROOT / "results" / SCENARIO
print("Scenario:", SCENARIO, "| results ->", RESULTS)

Scenario: multi | results -> /kaggle/working/results/multi


## 3. Pipeline definitions

In [3]:
REFERENCE = dict(bg="kept", size="letterbox", res=224, aug="light", color="rgb")
CHANGES = {
    "1": {},
    "2": dict(size="stretch"),
    "3": dict(res=384),
    "4": dict(aug="none"),
    "5": dict(aug="strong"),
    "6": dict(color="gray"),
    "7": dict(color="hsv"),
    "8": dict(color="lab"),
    "9": dict(color="clahe"),
}
PIPELINE_DEFS = {}
for k, ch in CHANGES.items():
    PIPELINE_DEFS["K" + k] = {**REFERENCE, **ch, "bg": "kept", "balanced": False, "tta": False}
    PIPELINE_DEFS["R" + k] = {**REFERENCE, **ch, "bg": "removed", "balanced": False, "tta": False}
PIPELINE_DEFS["F"] = {**REFERENCE, "balanced": True, "tta": True}   # placeholder, replaced in section 7
display(pd.DataFrame(PIPELINE_DEFS).T.drop(index="F"))

,bg,size,res,aug,color,balanced,tta
K1,kept,letterbox,224,light,rgb,False,False
R1,removed,letterbox,224,light,rgb,False,False
K2,kept,stretch,224,light,rgb,False,False
R2,removed,stretch,224,light,rgb,False,False
K3,kept,letterbox,384,light,rgb,False,False
R3,removed,letterbox,384,light,rgb,False,False
K4,kept,letterbox,224,none,rgb,False,False
R4,removed,letterbox,224,none,rgb,False,False
K5,kept,letterbox,224,strong,rgb,False,False
R5,removed,letterbox,224,strong,rgb,False,False


## 4. Find the inputs

Finds the Stage 0 output (`stage0/metadata/manifest.csv`), the Notebook 1 output (`bg_removed/mask_manifest.csv`) and, if attached, results from earlier versions of this notebook.

In [4]:
def walk_find(base, predicate, skip=("standardized", "cutouts", "masks", "figures", "weights")):
    for dirpath, dirnames, filenames in os.walk(base, followlinks=True):
        d = Path(dirpath)
        hit = predicate(d, filenames)
        if hit:
            return hit
        dirnames[:] = [x for x in dirnames if x not in skip]
    return None

STAGE0 = walk_find(INPUT_ROOT, lambda d, f: d.parent if ("manifest.csv" in f and d.name == "metadata"
                                                        and d.parent.name == "stage0") else None)
assert STAGE0, "Stage 0 output not found. Add Input -> Your Work -> your Stage 0 notebook."
manifest = pd.read_csv(STAGE0 / "metadata" / "manifest.csv", dtype={"image_id": str})
CLASS_MAP = json.loads((STAGE0 / "metadata" / "class_map.json").read_text())
CLASSES = sorted(CLASS_MAP, key=CLASS_MAP.get)
NUM_CLASSES = len(CLASSES)
# capture device: from the manifest (new Stage 0) or, for older Stage 0 outputs, from the camera EXIF in inventory.csv
DEVICE_NAMES = {"apple iphone 14": "iPhone 14", "xiaomi redmi note 15 5g": "Redmi Note 15", "samsung galaxy m14": "Galaxy M14"}
NO_EXIF_DEVICE = "WhatsApp (no camera EXIF)"
def add_device(man, stage0):
    if "device" in man:
        return man, "manifest"
    inv = stage0 / "metadata" / "inventory.csv"
    if inv.exists():
        cam = pd.read_csv(inv, dtype={"image_id": str}, usecols=lambda c: c in ("image_id", "camera"))
        if "camera" in cam:
            cam = cam.drop_duplicates("image_id")
            def dev(c):
                c2 = " ".join(str(c).lower().split()) if isinstance(c, str) else ""
                return DEVICE_NAMES.get(c2, str(c).strip()) if c2 else NO_EXIF_DEVICE
            man = man.merge(cam, on="image_id", how="left")
            man["device"] = man.camera.map(dev)
            return man.drop(columns="camera"), "inventory.csv (camera EXIF)"
    man["device"] = "unknown"
    return man, "not available"
manifest["y"] = manifest.label.map(CLASS_MAP)
manifest, dev_src = add_device(manifest, STAGE0)
print("Stage 0:", STAGE0, "|", len(manifest), "images |", NUM_CLASSES, "classes:", CLASSES)
print("Device column from:", dev_src); print(manifest.device.value_counts().to_string())

BGR = walk_find(INPUT_ROOT, lambda d, f: d if ("mask_manifest.csv" in f and d.name == "bg_removed") else None)
needs_removed = True   # the plan contains R pipelines
if BGR:
    mm = pd.read_csv(BGR / "mask_manifest.csv", dtype={"image_id": str})
    manifest = manifest.merge(mm[["image_id", "cutout_path", "mask_status"]], on="image_id", how="left")
    missing = manifest.cutout_path.isna().sum()
    print("Notebook 1:", BGR, "| cut-outs:", manifest.cutout_path.notna().sum(), "of", len(manifest))
    assert missing == 0, (f"{missing} Stage 0 images have no cut-out. Notebook 1 was run on a different set of images: "
                          "rerun Notebook 1 on this Stage 0 output.")
    print("Background-removal fallbacks (original image kept), per class:")
    display(manifest[manifest.mask_status != "ok"].label.value_counts().reindex(CLASSES, fill_value=0).to_frame("images").T)
else:
    assert not needs_removed, "R pipelines need the Notebook 1 output. Add Input -> Your Work -> Notebook 1."
    print("Notebook 1 output not attached (only needed for R pipelines).")

# carry forward results from earlier versions of this notebook (every scenario found, so the latest
# version always holds all finished scenarios; only the current scenario's folder is written to)
RESULTS.mkdir(parents=True, exist_ok=True)
for sc in ("all", "single", "multi"):
    prev = walk_find(INPUT_ROOT, lambda d, f, sc=sc: d if ("summary_all.csv" in f and d.name == sc
                                                          and d.parent.name == "results") else None,
                     skip=("standardized", "cutouts", "masks", "figures", "weights"))
    dest = OUT_ROOT / "results" / sc
    if prev and prev.resolve() != dest.resolve():
        n = 0
        for src in prev.rglob("*"):
            dst = dest / src.relative_to(prev)
            if src.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst); n += 1
        print(f"Carried forward {n} result files of scenario '{sc}' from {prev}")
# all commits and scenarios must use the same Stage 0 split, otherwise results cannot be compared
import hashlib
SPLIT_SIG = hashlib.sha1(manifest.sort_values("image_id")[["image_id", "split", "fold"]]
                         .to_csv(index=False).encode()).hexdigest()[:16]
for sc in ("all", "single", "multi"):
    f = OUT_ROOT / "results" / sc / "split_signature.txt"
    if f.exists() and f.read_text().strip() != SPLIT_SIG:
        raise SystemExit(f"STOP: results of scenario '{sc}' were made with a different Stage 0 split. "
                         "Attach the same Stage 0 output that the earlier commits used.")
(RESULTS / "split_signature.txt").write_text(SPLIT_SIG)
print("Stage 0 split signature:", SPLIT_SIG)
if (RESULTS / "summary_all.csv").exists():
    done = pd.read_csv(RESULTS / "summary_all.csv")
    print("Already finished:", sorted(done.drop_duplicates(["model", "pipeline"]).apply(lambda r: f"{r.model}/{r.pipeline}", axis=1)))

full_manifest = manifest
if SCENARIO != "all":
    manifest = full_manifest[full_manifest.subset == SCENARIO].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
tv_df = manifest[manifest.split == "trainval"].reset_index(drop=True)
# cross-test set: the test images of the other subset (single <-> multi)
OTHER = {"single": "multi", "multi": "single"}.get(SCENARIO)
cross_subset_df = (full_manifest[(full_manifest.split == "test") & (full_manifest.subset == OTHER)].reset_index(drop=True)
                   if OTHER else None)
if SCENARIO != "all":   # the scenario's own images only; the other subset is used only for the cross-test
    assert (tv_df.subset == SCENARIO).all() and (test_df.subset == SCENARIO).all()
    assert (cross_subset_df.subset == OTHER).all() and not set(cross_subset_df.image_id) & set(manifest.image_id)
print(f"Scenario {SCENARIO}: {len(tv_df)} train/val images, {len(test_df)} test images"
      + (f", {len(cross_subset_df)} {OTHER} test images for the cross-test" if OTHER else ""))
display(pd.crosstab(manifest.split.where(manifest.split == "test", "fold " + manifest.fold.astype(str)),
                    manifest.label).reindex(columns=CLASSES))

Stage 0: /kaggle/input/notebooks/arnicasarker/stage0-single-multi-3parts/stage0 | 2950 images | 6 classes: ['Churi', 'Dao', 'Khor', 'Kodal', 'Kural', 'Shabol']
Device column from: inventory.csv (camera EXIF)
device
iPhone 14                    1119
Redmi Note 15                 647
WhatsApp (no camera EXIF)     646
Galaxy M14                    538
Notebook 1: /kaggle/input/notebooks/arnicasarker/nb1-background-removal/bg_removed | cut-outs: 2950 of 2950
Background-removal fallbacks (original image kept), per class:


label,Churi,Dao,Khor,Kodal,Kural,Shabol
images,2,5,0,22,20,45


Carried forward 313 result files of scenario 'all' from /kaggle/input/notebooks/arnicasarker/nb2-training-scenarios/results/all
Carried forward 332 result files of scenario 'single' from /kaggle/input/notebooks/arnicasarker/nb2-training-scenarios/results/single
Stage 0 split signature: 87e5e3e1f4bc1271
Scenario multi: 1010 train/val images, 164 test images, 255 single test images for the cross-test


label,Churi,Dao,Khor,Kodal,Kural,Shabol
split,,,,,,
fold 0,40,37,12,31,29,35
fold 1,33,39,34,35,28,36
fold 2,36,39,44,33,29,42
fold 3,39,35,14,32,34,46
fold 4,42,37,23,29,32,35
test,31,36,12,27,27,31


## 5. Image preparation

Deterministic steps (background choice, CLAHE, resize) are done once per pipeline and cached in memory. Random augmentation and the color conversion are applied on the fly during training.

In [5]:
PAD_GRAY = 128   # border colour for letterbox and for rotations (same gray as the cut-out background)

def load_rgb(row, bg):
    path = (BGR / row.cutout_path) if bg == "removed" else (STAGE0 / row.path)
    return np.asarray(Image.open(path).convert("RGB"))

def clahe_rgb(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def geometric(rgb, size, res):
    h, w = rgb.shape[:2]
    if size == "stretch":
        return cv2.resize(rgb, (res, res), interpolation=cv2.INTER_AREA)
    s = res / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    small = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    out = np.full((res, res, 3), PAD_GRAY, np.uint8)
    top, left = (res - nh) // 2, (res - nw) // 2
    out[top:top + nh, left:left + nw] = small
    return out

def build_cache(df, cfg):
    arrs = []
    for r in tqdm(df.itertuples(index=False), total=len(df), desc=f"preparing {cfg['bg']} {cfg['size']} {cfg['res']}", leave=False):
        rgb = load_rgb(r, cfg["bg"])
        if cfg["color"] == "clahe":
            rgb = clahe_rgb(rgb)
        arrs.append(geometric(rgb, cfg["size"], cfg["res"]))
    return np.stack(arrs)

def convert_color(rgb, color):
    # rgb: HxWx3 uint8 -> HxWx3 uint8 in the chosen representation
    if color == "gray":
        g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        return np.repeat(g[..., None], 3, axis=2)
    if color == "hsv":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV_FULL)
    if color == "lab":
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    return rgb   # "rgb" and "clahe" (already applied in the cache)

IMAGENET_MEAN, IMAGENET_STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def channel_stats(cache, color):
    # ImageNet statistics for RGB-like inputs; own statistics (from training images) for HSV and LAB
    if color in ("rgb", "clahe", "gray"):
        return IMAGENET_MEAN, IMAGENET_STD
    x = np.stack([convert_color(im, color) for im in cache[:: max(1, len(cache) // 300)]]).astype(np.float32) / 255
    return x.mean(axis=(0, 1, 2)), x.std(axis=(0, 1, 2)) + 1e-6

class RandomJPEG:
    # re-encode at a random JPEG quality (simulates phone and messenger compression); x: uint8 CxHxW tensor
    def __init__(self, p=0.5, quality=(20, 95)):
        self.p, self.q = p, quality
    def __call__(self, x):
        if random.random() >= self.p:
            return x
        a = x.permute(1, 2, 0).numpy()
        ok, buf = cv2.imencode(".jpg", a[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, random.randint(*self.q)])
        return torch.from_numpy(cv2.imdecode(buf, cv2.IMREAD_COLOR)[..., ::-1].copy()).permute(2, 0, 1)

class RandomWhiteBalance:
    # random per-channel gain (simulates different camera colour processing)
    def __init__(self, p=0.5, spread=0.12):
        self.p, self.s = p, spread
    def __call__(self, x):
        if random.random() >= self.p:
            return x
        g = torch.tensor([random.uniform(1 - self.s, 1 + self.s) for _ in range(3)]).view(3, 1, 1)
        return (x.float() * g).clamp(0, 255).to(torch.uint8)

def train_augment(aug):
    fill = [PAD_GRAY] * 3
    if aug == "none":
        return T.Identity()
    if aug == "light":
        return T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(15, fill=fill),
                          T.ColorJitter(brightness=0.2, contrast=0.2)])
    if aug == "strong":
        return T.Compose([T.RandomHorizontalFlip(), T.RandAugment(num_ops=2, magnitude=9, fill=fill)])
    if aug == "camera":   # light augmentation + camera simulation (colour, white balance, blur, sharpness, JPEG)
        return T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(15, fill=fill),
                          T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
                          RandomWhiteBalance(p=0.5),
                          T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),
                          T.RandomAdjustSharpness(sharpness_factor=2.0, p=0.3),
                          RandomJPEG(p=0.5, quality=(20, 95))])
    raise ValueError(aug)

class ImageSet(Dataset):
    def __init__(self, cache, labels, color, mean, std, augment=None, erase=False, hflip=False):
        self.cache, self.labels, self.color = cache, labels, color
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)
        self.augment, self.hflip = augment, hflip
        self.erase = T.RandomErasing(p=0.25, value=0) if erase else None
    def __len__(self):
        return len(self.cache)
    def __getitem__(self, i):
        x = torch.from_numpy(self.cache[i]).permute(2, 0, 1)            # uint8 CxHxW
        if self.augment is not None:
            x = self.augment(x)
        if self.hflip:
            x = torch.flip(x, dims=[2])
        x = convert_color(x.permute(1, 2, 0).numpy().copy(), self.color)
        x = (torch.from_numpy(x).permute(2, 0, 1).float() / 255 - self.mean) / self.std
        if self.erase is not None:
            x = self.erase(x)
        return x, int(self.labels[i])

## 6. Training and evaluation

In [6]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_model(res):
    kw = dict(pretrained=PRETRAINED, num_classes=NUM_CLASSES)
    if MODEL_NAME.startswith(("vit", "deit", "swin")):
        kw["img_size"] = res
    return timm.create_model(MODEL_NAME, **kw)

def predict(model, cache, labels, color, mean, std, tta):
    model.eval()
    views = [False, True] if tta else [False]
    probs = 0
    for flip in views:
        dl = DataLoader(ImageSet(cache, labels, color, mean, std, hflip=flip), batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=NUM_WORKERS)
        out = []
        with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            for x, _ in dl:
                out.append(F_.softmax(model(x.to(DEVICE)).float(), dim=1).cpu())
        probs = probs + torch.cat(out).numpy()
    return probs / len(views)

def train_fold(cfg, cache_tr, y_tr, cache_va, y_va, mean, std, seed):
    seed_everything(seed)
    ds = ImageSet(cache_tr, y_tr, cfg["color"], mean, std, augment=train_augment(cfg["aug"]),
                  erase=(cfg["aug"] == "strong"))
    if cfg["balanced"]:
        counts = np.bincount(y_tr, minlength=NUM_CLASSES)
        w = 1.0 / counts[y_tr]
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(y_tr), replacement=True)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True)
    else:
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    cutmix = T.CutMix(num_classes=NUM_CLASSES, alpha=1.0) if cfg["aug"] == "strong" else None

    model = make_model(cfg["res"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total = EPOCHS * len(dl); warm = WARMUP_EPOCHS * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    log = []
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot, n = 0.0, 0
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if cutmix is not None and random.random() < 0.5:
                x, y = cutmix(x, y)
            with torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x); n += len(x)
        log.append({"epoch": ep + 1, "train_loss": tot / max(n, 1), "sec": round(time.time() - t0, 1)})
    p_va = predict(model, cache_va, y_va, cfg["color"], mean, std, cfg["tta"])
    f1 = f1_score(y_va, p_va.argmax(1), average="macro")
    log[-1]["val_macro_f1"] = f1
    return model, p_va, pd.DataFrame(log)

def metrics(y, p):
    pred = p.argmax(1)
    return {"macro_f1": f1_score(y, pred, average="macro"), "accuracy": accuracy_score(y, pred),
            "balanced_acc": balanced_accuracy_score(y, pred)}

def save_confusion(y, pred, title, path):
    cm = confusion_matrix(y, pred, labels=range(NUM_CLASSES))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(NUM_CLASSES), CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

def pred_frame(df, probs):
    return pd.DataFrame({"image_id": df.image_id.to_numpy(), "subset": df.subset.to_numpy(),
                         "device": df.device.to_numpy(), "y": df.y.to_numpy(), "pred": probs.argmax(1),
                         **{f"p_{c}": probs[:, i] for i, c in enumerate(CLASSES)}})

def run_pipeline(name, cfg=None, tv=None, te=None, extra=None):
    # name: folder name; cfg: pipeline settings (default PIPELINE_DEFS[name]);
    # tv / te: train-val and test tables (default: the scenario's); extra: {label: table} predicted by the test ensemble
    cfg = cfg or PIPELINE_DEFS[name]
    tv = tv_df if tv is None else tv.reset_index(drop=True)
    te = test_df if te is None else te.reset_index(drop=True)
    extra = extra or {}
    out = RESULTS / MODEL_NAME / name
    if (out / "summary.json").exists():
        print(f"{MODEL_NAME}/{name}: already done, skipping"); return
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {SCENARIO} | {MODEL_NAME} / {name}: {cfg} ===")
    t_start = time.time()
    cache_tv, cache_te = build_cache(tv, cfg), build_cache(te, cfg)
    cache_ex = {k: build_cache(d, cfg) for k, d in extra.items()}
    y_tv, y_te = tv.y.to_numpy(), te.y.to_numpy()

    oof = np.zeros((len(tv), NUM_CLASSES)); test_probs = 0; fold_rows, logs = [], []
    ex_probs = {k: 0 for k in extra}
    norm = {}
    for k in FOLDS:
        tr, va = np.flatnonzero(tv.fold != k), np.flatnonzero(tv.fold == k)
        mean, std = channel_stats(cache_tv[tr], cfg["color"])
        norm[str(k)] = {"mean": [float(v) for v in mean], "std": [float(v) for v in std]}
        model, p_va, log = train_fold(cfg, cache_tv[tr], y_tv[tr], cache_tv[va], y_tv[va], mean, std, SEED + k)
        oof[va] = p_va
        test_probs = test_probs + predict(model, cache_te, y_te, cfg["color"], mean, std, cfg["tta"])
        for key, d in extra.items():
            ex_probs[key] = ex_probs[key] + predict(model, cache_ex[key], d.y.to_numpy(), cfg["color"], mean, std, cfg["tta"])
        m = metrics(y_tv[va], p_va); m.update(fold=k, n_val=len(va)); fold_rows.append(m)
        logs.append(log.assign(fold=k))
        print(f"  fold {k}: macro-F1 {m['macro_f1']:.3f} | acc {m['accuracy']:.3f} | "
              f"{log.sec.sum() / 60:.1f} min")
        if k == FOLDS[0] or name in SAVE_ALL_FOLD_WEIGHTS:   # weights for Grad-CAM and robustness in Notebook 3
            (out / "weights").mkdir(exist_ok=True)
            torch.save({k2: v.half() for k2, v in model.state_dict().items()}, out / "weights" / f"fold{k}.pt")
        del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    test_probs = test_probs / len(FOLDS)
    (out / "weights").mkdir(exist_ok=True)
    (out / "weights" / "norm.json").write_text(json.dumps(norm, indent=2))   # input normalisation per fold

    folds = pd.DataFrame(fold_rows); folds.to_csv(out / "fold_metrics.csv", index=False)
    pd.concat(logs).to_csv(out / "train_log.csv", index=False)
    in_folds = tv.fold.isin(FOLDS).to_numpy()
    pred_frame(tv, oof).assign(fold=tv.fold.to_numpy())[in_folds].to_csv(out / "oof_predictions.csv", index=False)
    pred_frame(te, test_probs).to_csv(out / "test_predictions.csv", index=False)
    extra_summary = {}
    for key, d in extra.items():
        p = ex_probs[key] / len(FOLDS)
        pred_frame(d, p).to_csv(out / f"{key}_predictions.csv", index=False)
        mk = metrics(d.y.to_numpy(), p)
        extra_summary.update({f"{key}_macro_f1": mk["macro_f1"], f"{key}_accuracy": mk["accuracy"], f"{key}_n": len(d)})
        save_confusion(d.y.to_numpy(), p.argmax(1), f"{MODEL_NAME} {name}: {key}", out / f"confusion_{key}.png")
    oof_m, test_m = metrics(y_tv[in_folds], oof[in_folds]), metrics(y_te, test_probs)
    per_class = f1_score(y_te, test_probs.argmax(1), average=None, labels=range(NUM_CLASSES))
    save_confusion(y_tv[in_folds], oof[in_folds].argmax(1), f"{MODEL_NAME} {name}: cross-validation", out / "confusion_cv.png")
    save_confusion(y_te, test_probs.argmax(1), f"{MODEL_NAME} {name}: test set", out / "confusion_test.png")

    summary = {"scenario": SCENARIO, "model": MODEL_NAME, "pipeline": name, **{f"cfg_{k}": v for k, v in cfg.items()},
               "cv_macro_f1_mean": folds.macro_f1.mean(), "cv_macro_f1_std": folds.macro_f1.std(ddof=1) if len(folds) > 1 else 0.0,
               "cv_accuracy_mean": folds.accuracy.mean(), "oof_macro_f1": oof_m["macro_f1"],
               "test_macro_f1": test_m["macro_f1"], "test_accuracy": test_m["accuracy"],
               "test_balanced_acc": test_m["balanced_acc"],
               **{f"test_f1_{c}": float(v) for c, v in zip(CLASSES, per_class)}, **extra_summary,
               "n_trainval": len(tv), "n_test": len(te),
               "epochs": EPOCHS, "folds": len(FOLDS), "minutes": round((time.time() - t_start) / 60, 1)}
    (out / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    all_p = RESULTS / "summary_all.csv"
    row = pd.DataFrame([summary])
    (pd.concat([pd.read_csv(all_p), row]) if all_p.exists() else row).to_csv(all_p, index=False)
    print(f"  => CV macro-F1 {summary['cv_macro_f1_mean']:.3f} +/- {summary['cv_macro_f1_std']:.3f} | "
          f"test macro-F1 {summary['test_macro_f1']:.3f}"
          + "".join(f" | {k.replace('_macro_f1', '')} macro-F1 {v:.3f}" for k, v in extra_summary.items() if k.endswith("macro_f1"))
          + f" | {summary['minutes']} min")

## 7. Run the plan

After the plan, scenario `all` adds two cross-device runs. The model with the best CV macro-F1 on F is retrained on **iPhone images only** (same folds, same settings): `F_iphone` with the F settings, `F_iphone_bgremoved` with background-removed images, and `F_iphone_camaug` with camera-simulation augmentation (random JPEG compression, blur, sharpening, colour and white-balance changes) to test whether simulated camera variation reduces the gap. Its test set is the iPhone part of the locked test set (same phone), and every image from the other phones is predicted as an unseen-device set (`cross_device_predictions.csv`). None of those images is used in training.

F is chosen from the ResNet-50 results: for each factor, the level with the highest CV macro-F1, averaged over the kept (K) and removed (R) versions; background by the mean over all K vs. all R pipelines. The choice is saved to `results/f_settings.json` and reused by every later commit and model.

In [7]:
SIZE_OPT  = {"1": ("letterbox", 224), "2": ("stretch", 224), "3": ("letterbox", 384)}
AUG_OPT   = {"1": "light", "4": "none", "5": "strong"}
COLOR_OPT = {"1": "rgb", "6": "gray", "7": "hsv", "8": "lab", "9": "clahe"}

def choose_f_settings():
    f_json = RESULTS / "f_settings.json"
    if f_json.exists():
        return json.loads(f_json.read_text())
    if F_SETTINGS is not None:
        fs = dict(F_SETTINGS)
    else:
        if not (RESULTS / "summary_all.csv").exists():
            return None
        res = pd.read_csv(RESULTS / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last")
        r = res[res.model == STAGE1_MODEL].set_index("pipeline").cv_macro_f1_mean
        if not all(f"{b}{i}" in r.index for b in "KR" for i in range(1, 10)):
            return None
        score = lambda i: (r[f"K{i}"] + r[f"R{i}"]) / 2
        k_mean = r[[f"K{i}" for i in range(1, 10)]].mean(); r_mean = r[[f"R{i}" for i in range(1, 10)]].mean()
        best = lambda opts: opts[max(opts, key=score)]
        size, res_px = best(SIZE_OPT)
        fs = dict(bg="kept" if k_mean >= r_mean else "removed", size=size, res=res_px,
                  aug=best(AUG_OPT), color=best(COLOR_OPT))
        print(f"Background: K mean {k_mean:.3f} vs R mean {r_mean:.3f}")
        for title, opts in [("size", SIZE_OPT), ("augmentation", AUG_OPT), ("color", COLOR_OPT)]:
            print(f"{title:13s}", " | ".join(f"{opts[i]}: {score(i):.3f}" for i in opts))
    f_json.write_text(json.dumps(fs, indent=2))
    print("F settings:", fs)
    return fs

# rough relative cost per pipeline (ResNet-50 at 224 px = 1)
MODEL_COST = {"resnet50": 1.0, "convnext_tiny": 1.3, "efficientnet_b0": 0.8, "vit_small_patch16_224": 0.8}
def cost(model, cfg):
    hi = cfg["res"] >= 384
    return MODEL_COST.get(model, 1.0) * ((3.5 if model.startswith("vit") else 2.5) if hi else 1.0) * (1.15 if cfg["tta"] else 1.0)

T_SESSION = time.time()
unit_min = None          # minutes per unit of cost, measured on this GPU
for MODEL_NAME, name in PLAN:
    if (RESULTS / MODEL_NAME / name / "summary.json").exists():
        continue
    if name == "F":
        fs = choose_f_settings()
        if fs is None:
            print(f"{MODEL_NAME}/F waits until all K and R pipelines are finished."); continue
        PIPELINE_DEFS["F"] = {**fs, "balanced": True, "tta": True}
    c = cost(MODEL_NAME, PIPELINE_DEFS[name]) * len(tv_df) / 2500
    elapsed_h = (time.time() - T_SESSION) / 3600
    expected_h = (unit_min or 0) * c / 60
    if elapsed_h + expected_h > TIME_BUDGET_HOURS:
        print(f"Time budget reached ({elapsed_h:.1f} h used). Next: {MODEL_NAME}/{name} (about {expected_h:.1f} h).")
        break
    t0 = time.time()
    run_pipeline(name, extra={"cross_subset": cross_subset_df} if OTHER else None)
    unit_min = (time.time() - t0) / 60 / c

# ---- scenario "all": cross-device run (best model on F, trained on one phone only) ----
def best_on_f():
    if not (RESULTS / "summary_all.csv").exists():
        return None
    res = pd.read_csv(RESULTS / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last")
    f = res[(res.pipeline == "F") & res.model.isin(MODELS)]
    return f.sort_values("cv_macro_f1_mean").model.iloc[-1] if set(f.model) >= set(MODELS) else None

# cross-device variants: same model and F settings, trained on one phone only; the second variant uses
# background-removed images, to test whether the scene/background is what fails to transfer between phones
CROSS_DEVICE_VARIANTS = {"F_iphone": {}, "F_iphone_bgremoved": {"bg": "removed"}, "F_iphone_camaug": {"aug": "camera"}}
EXTRA_STEPS = []
if SCENARIO == "all" and (manifest.device == CROSS_DEVICE_TRAIN).any():
    EXTRA_STEPS = list(CROSS_DEVICE_VARIANTS)
    bm = best_on_f()
    if bm is None:
        print("Cross-device runs wait until all models are finished on F.")
    else:
        MODEL_NAME = bm
        fs = choose_f_settings()
        tv_i = tv_df[tv_df.device == CROSS_DEVICE_TRAIN]
        for vname, change in CROSS_DEVICE_VARIANTS.items():
            if (RESULTS / bm / vname / "summary.json").exists():
                continue
            cfg = {**fs, **change, "balanced": True, "tta": True}
            c = cost(MODEL_NAME, cfg) * len(tv_i) / 2500
            elapsed_h = (time.time() - T_SESSION) / 3600
            if elapsed_h + (unit_min or 0) * c / 60 > TIME_BUDGET_HOURS:
                print(f"Time budget reached ({elapsed_h:.1f} h used). Next: {bm}/{vname}."); break
            t0 = time.time()
            run_pipeline(vname, cfg=cfg, tv=tv_i, te=test_df[test_df.device == CROSS_DEVICE_TRAIN],
                         extra={"cross_device": manifest[manifest.device != CROSS_DEVICE_TRAIN].reset_index(drop=True)})
            unit_min = (time.time() - t0) / 60 / c
elif SCENARIO == "all":
    print(f"No '{CROSS_DEVICE_TRAIN}' images in the manifest: cross-device run skipped.")

# ---- foundation-model baseline: frozen DINOv2 features + logistic regression (linear probe) ----
# "LP": all images of the scenario, same folds and test set as every other run (baseline for the model comparison)
# "LP_iphone" (scenario all): trained on iPhone images only, tested on the unseen phones like F_iphone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
LP_MODEL = "vit_small_patch14_dinov2.lvd142m"
LP_CFG = dict(bg="kept", size="letterbox", res=224, aug="none", color="rgb", balanced=True, tta=True)
_feat_cache = {}

def lp_features(df):
    missing = [i for i in df.image_id if i not in _feat_cache]
    if missing:
        net = timm.create_model(LP_MODEL, pretrained=True, num_classes=0, img_size=LP_CFG["res"]).eval().to(DEVICE)
        sub = df[df.image_id.isin(missing)].drop_duplicates("image_id").reset_index(drop=True)
        cache = build_cache(sub, LP_CFG)
        feats = 0
        for flip in (False, True):      # average the features of the image and its mirror
            dl = DataLoader(ImageSet(cache, np.zeros(len(sub), int), "rgb", IMAGENET_MEAN, IMAGENET_STD, hflip=flip),
                            batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
            out = []
            with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
                for x, _ in dl:
                    out.append(net(x.to(DEVICE)).float().cpu())
            feats = feats + torch.cat(out).numpy()
        for i, f in zip(sub.image_id, feats / 2):
            _feat_cache[i] = f
        del net; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return np.stack([_feat_cache[i] for i in df.image_id])

def run_linear_probe(name, tv=None, te=None, extra=None):
    tv = tv_df if tv is None else tv.reset_index(drop=True)
    te = test_df if te is None else te.reset_index(drop=True)
    extra = extra or {}
    out = RESULTS / LP_MODEL / name
    if (out / "summary.json").exists():
        return
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {SCENARIO} | {LP_MODEL} (frozen) + logistic regression / {name} ===")
    t_start = time.time()
    X_tv, X_te = lp_features(tv), lp_features(te)
    X_ex = {k: lp_features(d) for k, d in extra.items()}
    y_tv, y_te = tv.y.to_numpy(), te.y.to_numpy()
    oof = np.zeros((len(tv), NUM_CLASSES)); test_probs = 0; ex_probs = {k: 0 for k in extra}; fold_rows = []
    def proba(clf, sc, X):
        p = np.zeros((len(X), NUM_CLASSES)); p[:, clf.classes_] = clf.predict_proba(sc.transform(X)); return p
    for k in FOLDS:
        tr, va = np.flatnonzero(tv.fold != k), np.flatnonzero(tv.fold == k)
        sc = StandardScaler().fit(X_tv[tr])
        clf = LogisticRegression(max_iter=5000, C=1.0, class_weight="balanced").fit(sc.transform(X_tv[tr]), y_tv[tr])
        oof[va] = proba(clf, sc, X_tv[va])
        test_probs = test_probs + proba(clf, sc, X_te)
        for key in extra:
            ex_probs[key] = ex_probs[key] + proba(clf, sc, X_ex[key])
        m = metrics(y_tv[va], oof[va]); m.update(fold=k, n_val=len(va)); fold_rows.append(m)
        print(f"  fold {k}: macro-F1 {m['macro_f1']:.3f} | acc {m['accuracy']:.3f}")
    test_probs = test_probs / len(FOLDS)
    folds = pd.DataFrame(fold_rows); folds.to_csv(out / "fold_metrics.csv", index=False)
    pred_frame(tv, oof).assign(fold=tv.fold.to_numpy()).to_csv(out / "oof_predictions.csv", index=False)
    pred_frame(te, test_probs).to_csv(out / "test_predictions.csv", index=False)
    extra_summary = {}
    for key, d in extra.items():
        p = ex_probs[key] / len(FOLDS)
        pred_frame(d, p).to_csv(out / f"{key}_predictions.csv", index=False)
        mk = metrics(d.y.to_numpy(), p)
        extra_summary.update({f"{key}_macro_f1": mk["macro_f1"], f"{key}_accuracy": mk["accuracy"], f"{key}_n": len(d)})
    test_m = metrics(y_te, test_probs)
    per_class = f1_score(y_te, test_probs.argmax(1), average=None, labels=range(NUM_CLASSES))
    summary = {"scenario": SCENARIO, "model": LP_MODEL, "pipeline": name, **{f"cfg_{k}": v for k, v in LP_CFG.items()},
               "cv_macro_f1_mean": folds.macro_f1.mean(), "cv_macro_f1_std": folds.macro_f1.std(ddof=1),
               "cv_accuracy_mean": folds.accuracy.mean(), "oof_macro_f1": metrics(y_tv, oof)["macro_f1"],
               "test_macro_f1": test_m["macro_f1"], "test_accuracy": test_m["accuracy"], "test_balanced_acc": test_m["balanced_acc"],
               **{f"test_f1_{c}": float(v) for c, v in zip(CLASSES, per_class)}, **extra_summary,
               "n_trainval": len(tv), "n_test": len(te), "epochs": 0, "folds": len(FOLDS),
               "minutes": round((time.time() - t_start) / 60, 1)}
    (out / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    all_p = RESULTS / "summary_all.csv"
    row = pd.DataFrame([summary])
    (pd.concat([pd.read_csv(all_p), row]) if all_p.exists() else row).to_csv(all_p, index=False)
    print(f"  => CV macro-F1 {summary['cv_macro_f1_mean']:.3f} +/- {summary['cv_macro_f1_std']:.3f} | "
          f"test macro-F1 {summary['test_macro_f1']:.3f}"
          + "".join(f" | {k.replace('_macro_f1', '')} macro-F1 {v:.3f}" for k, v in extra_summary.items() if k.endswith("macro_f1"))
          + f" | {summary['minutes']} min")

LP_STEPS = ["LP"] + (["LP_iphone"] if SCENARIO == "all" and (manifest.device == CROSS_DEVICE_TRAIN).any() else [])
if (time.time() - T_SESSION) / 3600 < TIME_BUDGET_HOURS:
    run_linear_probe("LP", extra={"cross_subset": cross_subset_df} if OTHER else None)
    if "LP_iphone" in LP_STEPS:
        run_linear_probe("LP_iphone", tv=tv_df[tv_df.device == CROSS_DEVICE_TRAIN],
                         te=test_df[test_df.device == CROSS_DEVICE_TRAIN],
                         extra={"cross_device": manifest[manifest.device != CROSS_DEVICE_TRAIN].reset_index(drop=True)})

left = [f"{m}/{p}" for m, p in PLAN if not (RESULTS / m / p / "summary.json").exists()]
left += [f"{LP_MODEL}/{p}" for p in LP_STEPS if not (RESULTS / LP_MODEL / p / "summary.json").exists()]
bm = best_on_f() if EXTRA_STEPS else None
for vname in EXTRA_STEPS:
    if bm is None or not (RESULTS / bm / vname / "summary.json").exists():
        left.append(f"{bm or 'best model'}/{vname}")
print(f"\nSession time: {(time.time() - T_SESSION) / 3600:.2f} h")
if left:
    print(f"NOT FINISHED: {len(left)} step(s) left: {left}")
    print("-> Commit again with this notebook's latest version added as an input.")
else:
    done_sc = sorted(p.name for p in (OUT_ROOT / "results").iterdir() if (p / "summary_all.csv").exists())
    print(f"ALL DONE for scenario '{SCENARIO}'. Scenarios in this version's output: {done_sc}")
    print("-> Next: set SCENARIO to a scenario not yet done (attach this version as input), "
          "or run Notebook 3 once all three are done.")


=== multi | resnet50 / K1: {'bg': 'kept', 'size': 'letterbox', 'res': 224, 'aug': 'light', 'color': 'rgb', 'balanced': False, 'tta': False} ===


preparing kept letterbox 224:   0%|          | 0/1010 [00:00<?, ?it/s]

preparing kept letterbox 224:   0%|          | 0/164 [00:00<?, ?it/s]

preparing kept letterbox 224:   0%|          | 0/255 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

... (208 outputs left out)

preparing kept letterbox 224:   0%|          | 0/1010 [00:00<?, ?it/s]

preparing kept letterbox 224:   0%|          | 0/164 [00:00<?, ?it/s]

preparing kept letterbox 224:   0%|          | 0/255 [00:00<?, ?it/s]

  fold 0: macro-F1 1.000 | acc 1.000
  fold 1: macro-F1 0.981 | acc 0.980
  fold 2: macro-F1 0.989 | acc 0.991
  fold 3: macro-F1 0.986 | acc 0.985
  fold 4: macro-F1 0.989 | acc 0.990
  => CV macro-F1 0.989 +/- 0.007 | test macro-F1 0.988 | cross_subset macro-F1 0.836 | 2.0 min

Session time: 6.13 h
ALL DONE for scenario 'multi'. Scenarios in this version's output: ['all', 'multi', 'single']
-> Next: set SCENARIO to a scenario not yet done (attach this version as input), or run Notebook 3 once all three are done.


## 8. Results so far

In [8]:
res = pd.read_csv(RESULTS / "summary_all.csv").drop_duplicates(["model", "pipeline"], keep="last")
cols = ["scenario", "model", "pipeline", "cfg_bg", "cfg_size", "cfg_res", "cfg_aug", "cfg_color",
        "cv_macro_f1_mean", "cv_macro_f1_std", "test_macro_f1", "test_accuracy", "minutes"]
extra_cols = [c for c in res.columns if c.endswith("_macro_f1") and c.split("_")[0] in ("cross",)]
display(res[cols + extra_cols].sort_values(["model", "pipeline"]).round(3).reset_index(drop=True))
print("Results folder:", RESULTS)

,scenario,model,pipeline,cfg_bg,cfg_size,cfg_res,cfg_aug,cfg_color,cv_macro_f1_mean,cv_macro_f1_std,test_macro_f1,test_accuracy,minutes,cross_subset_macro_f1
0,multi,convnext_tiny,F,kept,letterbox,384,light,clahe,0.997,0.005,1.000,1.000,29.0,0.858
1,multi,convnext_tiny,K1,kept,letterbox,224,light,rgb,0.999,0.002,1.000,1.000,12.4,0.779
2,multi,convnext_tiny,R1,removed,letterbox,224,light,rgb,0.969,0.012,0.988,0.988,10.8,0.783
3,multi,efficientnet_b0,F,kept,letterbox,384,light,clahe,0.990,0.005,1.000,1.000,19.0,0.582
4,multi,efficientnet_b0,K1,kept,letterbox,224,light,rgb,0.964,0.007,0.995,0.994,9.3,0.552
5,multi,efficientnet_b0,R1,removed,letterbox,224,light,rgb,0.888,0.026,0.965,0.963,7.9,0.604
6,multi,resnet50,F,kept,letterbox,384,light,clahe,0.996,0.004,0.994,0.994,24.6,0.542
7,multi,resnet50,K1,kept,letterbox,224,light,rgb,0.996,0.002,0.989,0.988,13.8,0.534
8,multi,resnet50,K2,kept,stretch,224,light,rgb,0.995,0.005,0.995,0.994,10.3,0.561
9,multi,resnet50,K3,kept,letterbox,384,light,rgb,0.993,0.004,1.000,1.000,23.6,0.507


Results folder: /kaggle/working/results/multi


## 9. After each commit

- If section 7 printed **NOT FINISHED**: Add Input -> Your Work -> this notebook (latest version), then Save Version -> Save & Run All again. Nothing else to change.
- If it printed **ALL DONE**: this scenario is finished. Set `SCENARIO` to the next one and commit again (it starts a fresh plan in its own folder), or run it on a co-author's account.
- When all three scenarios are done: run Notebook 3 with the three outputs attached.

Do not change `EPOCHS`, `LR`, `BATCH_SIZE`, `SEED` or the plan between commits or scenarios, otherwise the comparison is no longer fair.

**Running the next scenario in the same notebook:** attach this notebook's latest version as an input, as for any later commit. Finished scenarios are copied along untouched (`results/all`, `results/single`, `results/multi`), so the latest version always holds everything for Notebook 3.